# 実行過程のビジュアライズ

この章では、`visualize_computational_process.py` を使って実行過程を確認します。
`profile` の集計値だけでは見えない命令スケジュールと依存関係を可視化します。
対応入力は **Dim2 単一チップ** の pipeline state JSON です。
PBC も `machine_type` が `Dim2` のため対象に含まれます。


In [1]:
import json
import pathlib
import os
from collections import Counter

from IPython.display import Code

project_root = pathlib.Path("../..").resolve()
qret_path = project_root / "build" / "main"
gridsynth_path = project_root / "externals" / "bin" / "gridsynth"
os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

## 0. 可視化入力を準備
可視化入力を用意するため、先に `Dim2` と `PBC` をコンパイルします。

> [!IMPORTANT]
> 
> `Dim3` / `DistributedDim2` / `DistributedDim3` はこの可視化ツールでは未対応です。


In [2]:
dim2_pipeline_path = "data/tutorial_5_dim2_pipeline.yaml"
pbc_pipeline_path = "data/tutorial_5_pbc_pipeline.yaml"

!qret compile --verbose --pipeline {dim2_pipeline_path}
!qret compile --verbose --pipeline {pbc_pipeline_path}

2026-03-02 14:02:30 - INFO  - Load OpenQASM2.
2026-03-02 14:02:30 - INFO  - Build IR from OpenQASM2.
2026-03-02 14:02:30 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-03-02 14:02:30 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-03-02 14:02:30 - INFO  - Run passes.
2026-03-02 14:02:30 - INFO  - Run InitCompileInfo
2026-03-02 14:02:30 - INFO  - Initialize compile information
2026-03-02 14:02:30 - INFO  - Run Mapping
2026-03-02 14:02:30 - INFO  - Run Routing
2026-03-02 14:02:30 - INFO  - Save SC_LS_FIXED_V0 pipeline state file.
2026-03-02 14:02:30 - INFO  - Saving SC_LS_FIXED_V0 pipeline state to JSON file: tutorial_5_dim2.json
2026-03-02 14:02:30 - INFO  - Load OpenQASM2.
2026-03-02 14:02:30 - INFO  - Build IR from OpenQASM2.
2026-03-02 14:02:30 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-03-02 14:02:30 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-03-02 14:02:30 - INFO  - Run passes.
2026-03-02 14:02:30 

## 1. 命令種別をざっくり比較

In [3]:
def count_types(path: str) -> Counter:
    data = json.loads(pathlib.Path(path).read_text())
    return Counter(inst["type"] for inst in data["program"])


for name, path in [("Dim2", "tutorial_5_dim2.json"), ("PBC", "tutorial_5_pbc.json")]:
    print(f"[{name}] {path}")
    for k, v in sorted(count_types(path).items()):
        print(f"  {k}: {v}")
    print()

[Dim2] tutorial_5_dim2.json
  ALLOCATE: 4
  ALLOCATE_MAGIC_FACTORY: 4
  DEALLOCATE: 4
  HADAMARD: 4
  LATTICE_SURGERY: 2
  LATTICE_SURGERY_MAGIC: 2
  PROBABILITY_HINT: 2
  TWIST: 2

[PBC] tutorial_5_pbc.json
  ALLOCATE: 12
  ALLOCATE_MAGIC_FACTORY: 4
  DEALLOCATE: 12
  LATTICE_SURGERY: 5
  MEAS_ZX: 9
  MOVE_MAGIC: 2
  TWIST: 4
  XOR: 2



## 2. ビジュアライザ起動
次のコマンドで可視化 UI を起動し、`tutorial_5_dim2.json` / `tutorial_5_pbc.json` を順に読み込みます。

```sh
streamlit run ../../visualizer/visualize_computational_process.py
```

主要モード:
1. `Single`: Timeline / Instructions / Dependency / Spatial 2D / Spatial 3D を 1 つずつ表示
2. `Unified`: Timeline + Instructions + Dependency + Spatial 2D を同時表示
3. `Playback`: 2D/3D を横並びで再生（`Start/Stop/Reset`, FPS）

見るポイント:
1. beat 順で並べた命令実行
2. 依存グラフの進行
3. チップ占有の差分（Dim2 と PBC）


## 3. PBC 観察ポイント
PBC では末尾測定を前提に命令系列が生成されます。
終盤の測定命令集中や古典依存（`XOR` など）の分布を重点的に確認すると、通常モードとの差分を把握しやすくなります。
